02 · Préparation des données et protocole d'évaluation

Étapes 3 et 4. Le code réutilisable est dans `src/inclusion_financiere/` (`data.py`, `features.py`, `evaluate.py`).

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inclusion_financiere.config import RANDOM_STATE, FIGURES, DECISIONS, COST_FP, COST_FN
from inclusion_financiere.data import load_raw, clean, split, make_cv, CAT_COLS, NUM_COLS

FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

Étape 3 · Séparer X et y, construire les variables

In [2]:
from inclusion_financiere.features import (AddFeatures, onehot_preprocessor, pipe,
                                           CAT_FE, NUM_FE, CAT_OWN, NUM_OWN)

X, y = clean(load_raw())
print("X :", X.shape, "| colonnes :", list(X.columns))
AddFeatures().fit_transform(X.head(3))[CAT_FE[-2:] + NUM_FE[2:]]

X : (23524, 10) | colonnes : ['country', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']


,age_group,household_group,education_rank,is_head,formal_job,urban_phone,is_alone
0,16-25,3-5,2,0,0,0,0
1,66+,3-5,0,1,0,0,0
2,26-35,3-5,3,0,0,1,0


Étape 4 · Mettre le test sous scellé

In [3]:
X_train, X_test, y_train, y_test = split(X, y)
print("X_train :", X_train.shape, "| prévalence", round(y_train.mean(), 4))
print("X_test  :", X_test.shape, "| prévalence", round(y_test.mean(), 4))
assert (len(X_train), len(X_test)) == (18819, 4705)   # point de contrôle
del X_test, y_test   # plus utilisés jusqu'à l'étape 10 (notebook 06)

X_train : (18819, 10) | prévalence 0.1408
X_test  : (4705, 10) | prévalence 0.1407


Ce que le préprocesseur apprend sur les données d'entraînement (ici ajusté une fois sur `X_train` pour l'inspecter ; en validation croisée il est réajusté dans chaque pli, via le pipeline) :

In [4]:
prep = pipe("passthrough")[:-1].fit(X_train, y_train)
ohe = prep[-1].named_transformers_["cat"]
print("Nombre de colonnes après encodage :", len(prep[-1].get_feature_names_out()))
for col, inf in zip(CAT_FE, ohe.infrequent_categories_):
    if inf is not None:
        print(f"Modalités regroupées en « infrequent » pour {col}: {list(inf)}")

Nombre de colonnes après encodage : 53
Modalités regroupées en « infrequent » pour marital_status: ['Dont know']


Q3.1 — `AddFeatures` n'apprend rien, ce sont des règles écrites à la main. Le `OneHotEncoder` apprend la liste des catégories, et le `StandardScaler` apprend la moyenne et l'écart-type. Si on les apprend sur toutes les données avant la validation croisée, le modèle a déjà « vu » les données de validation : le score est trop optimiste. Avec un pipeline, tout est réappris dans chaque pli, sur l'entraînement seulement.

Q3.2 — `min_frequency=20` : les modalités vues moins de 20 fois sont regroupées dans une seule colonne « infrequent ». `handle_unknown="infrequent_if_exist"` : si une modalité jamais vue arrive, elle va dans cette colonne au lieu de faire planter le code. Ça règle le problème de Q2.6.

Q3.3 —
- Régression logistique : le rang (`education_rank`) donne un seul effet « un niveau de plus » ; le one-hot donne un effet par niveau. Les deux ensemble se complètent.
- Arbre : le rang est plus pratique, une seule coupure (« rang ≥ 3 ») suffit pour séparer les niveaux élevés.

Q3.4 — Pour la forêt aléatoire, ça ne sert à rien : un arbre coupe selon un seuil, l'échelle ne change pas le résultat. Pour la régression logistique, c'est utile : le calcul converge mieux et la régularisation traite toutes les variables de la même façon (sinon l'âge, de 16 à 100, et `is_alone`, 0 ou 1, seraient pénalisés différemment).

Q3.5 — En plus des variables du guide, j'ai ajouté deux variables :
- `is_alone` : la personne vit seule. Elle gère son argent seule, donc elle a peut-être plus besoin d'un compte.
- `household_group` : la taille du ménage par tranches (0–2, 3–5, 6–9, 10+). Les grands ménages sont souvent plus pauvres.

On regarde si elles aident à l'étape 5 (Q5.2).

4.2 · Validation croisée stratifiée, un seul objet pour tous

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from inclusion_financiere.evaluate import summarize

cv = make_cv()
scoring = {"pr_auc": "average_precision", "roc_auc": "roc_auc",
           "brier": "neg_brier_score", "logloss": "neg_log_loss", "acc": "accuracy"}
logreg = pipe(LogisticRegression(max_iter=2000))
res = cross_validate(logreg, X_train, y_train, cv=cv, scoring=scoring,
                     return_train_score=True)
summarize(res, "logreg", list(scoring)).to_frame().T.set_index("model").round(4).T

model,logreg
pr_auc_val_mean,0.55948
pr_auc_val_std,0.028403
pr_auc_train_mean,0.565384
pr_auc_train_std,0.00665
roc_auc_val_mean,0.848999
roc_auc_val_std,0.01049
roc_auc_train_mean,0.852929
roc_auc_train_std,0.002527
brier_val_mean,0.087026
brier_val_std,0.003044


4.3 · Généraliser à un pays jamais vu

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
groups = X_train["country"]
pays, prev = [], []
for i, (tr, va) in enumerate(sgkf.split(X_train, y_train, groups)):
    print(i, list(groups.iloc[va].unique()), round(y_train.iloc[va].mean(), 3))
    pays.append(groups.iloc[va].unique()[0]); prev.append(y_train.iloc[va].mean())

res_pays = cross_validate(logreg, X_train, y_train, groups=groups, cv=sgkf,
                          scoring=["average_precision", "roc_auc"])
par_pays = pd.DataFrame({"pays": pays, "prevalence": prev,
                         "pr_auc": res_pays["test_average_precision"],
                         "roc_auc": res_pays["test_roc_auc"]})
par_pays["pr_auc / prevalence"] = par_pays["pr_auc"] / par_pays["prevalence"]
print(par_pays.round(3).to_string(index=False))
print(f"\nCV stratifiée : PR-AUC {res['test_pr_auc'].mean():.3f} ± {res['test_pr_auc'].std(ddof=1):.3f}"
      f" | ROC-AUC {res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std(ddof=1):.3f}")

0 ['Rwanda'] 0.116
1 ['Tanzania'] 0.092
2 ['Uganda'] 0.089
3 ['Kenya'] 0.248


    pays  prevalence  pr_auc  roc_auc  pr_auc / prevalence
  Rwanda       0.116   0.406    0.793                3.514
Tanzania       0.092   0.420    0.842                4.581
  Uganda       0.089   0.491    0.834                5.515
   Kenya       0.248   0.625    0.816                2.517

CV stratifiée : PR-AUC 0.559 ± 0.028 | ROC-AUC 0.849 ± 0.010


Q4.1 — La stratification garde la même part de « Yes » partout (0,1408 et 0,1407 ici). Avec seulement 2 % de « Yes », un petit pli pourrait n'en avoir aucun, et on ne pourrait pas calculer le recall ou la PR-AUC. Les scores dépendraient du hasard du découpage.

Q4.2 — Pour que tous les modèles soient testés sur les mêmes plis. Comme ça, si un modèle est meilleur, c'est grâce au modèle et pas grâce à un découpage plus facile. On peut aussi comparer les modèles pli par pli. Et les résultats sont reproductibles (graine 42).

Q4.3 — Il y a 86 % de « Non », donc la ROC-AUC paraît bonne facilement. La PR-AUC regarde seulement les « Yes » : parmi les gens contactés, combien ont un compte, et combien on en a trouvé. C'est la question de KaziPay. Au hasard, la PR-AUC vaut 0,141 ; la régression logistique fait 0,559, environ 4 fois mieux.

Q4.4 — Quand on teste sur un pays que le modèle n'a jamais vu, la ROC-AUC va de 0,793 (Rwanda) à 0,842 (Tanzanie). C'est un peu moins bien que la CV normale (0,849). Donc le modèle marche encore sur un nouveau pays, mais un peu moins bien.

Pour le Sénégal, ce n'est pas suffisant : les 4 pays sont tous en Afrique de l'Est et le Sénégal est différent. Il faudrait d'abord récupérer des données locales pour tester et réajuster le modèle.

Q4.5 — Non. `TimeSeriesSplit` sert quand on prédit le futur avec le passé. Ici, chaque personne est interrogée une seule fois, ce n'est pas une série dans le temps. Et l'année est la même chose que le pays (Q2.3), donc ce cas est déjà testé en 4.3.